In [ ]:
import numpy as np
import pandas as pd
import wfdb
#from extract import extract_to_df, extract_record
import yang.extract as ex
from load import load_dataset
import tqdm

import os

SEED = 42
np.random.seed(SEED)

In [ ]:
parent_dir = "c:/Users/wuyan/Projects/wfdb-python/"
datasets = ['mitdb', 'ltafdb', 'afdb']

dataset_name = datasets[2]
Extractor = ex.Extractor(parent_dir=parent_dir, dataset_name=dataset_name, wave_col='MLII')

record_names = Extractor.load_record_names()
print(f"Loading {len(record_names)} records from {dataset_name} dataset...")

In [ ]:
for record_name in tqdm.tqdm(record_names):


In [ ]:
datasets = ['mitdb', 'ltafdb', 'afdb']
directories = [f"c:/Users/wuyan/Projects/wfdb-python/{dataset}/" for dataset in datasets]

In [ ]:
i_dataset = 2
dataset_name = datasets[i_dataset]
directory = directories[i_dataset]

atr_files = [f for f in os.listdir(directory) if f.endswith('.atr')]
atr_file_names = [os.path.splitext(f)[0] for f in atr_files]

In [ ]:
i_file = 2
atr_file_name = atr_file_names[i_file]

In [ ]:
atr_file_name

In [ ]:
print(atr_file_name)
Extractor = ex.Extractor(directory)
df_wave = Extractor.extract_wave(atr_file_name, col_name = "ECG1") # assuming ECG1 is the MLII
df_ann = Extractor.extract_annotation(atr_file_name)

In [ ]:
# plot MLII vs Time with Go
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_wave['Time'], y=df_wave['MLII'], mode='lines', name='MLII'))
#fig.update_layout(title=title + f"for sample {idx} with label {label}", xaxis_title='Time (s)', yaxis_title='mV')
fig.show()


In [ ]:

Extractor = ex.Extractor(data_dir=directory)


Extractor.extract_record(atr_file_name)

In [ ]:
i_dataset = 2
i_file = 100
dataset_name = datasets[i_dataset]
data_dir = directories[i_dataset]
Extractor = ex.Extractor(data_dir=data_dir)
record = Extractor.extract_record(i_file)

df_wave, df_ann = Extractor.extract_to_df(i_file)

In [ ]:
# list all the .atr files in data_dir


# for each atr file, load the record and annotations calling the extract_record function
for atr_file in atr_file_names:
    record = wfdb.rdrecord(os.path.join(data_dir, atr_file))
    annotations = wfdb.rdann(os.path.join(data_dir, atr_file), 'atr')
    print(f"Record: {record.__dict__}")
    print(f"Annotations: {annotations.__dict__}")


### Check all Annotations

In [ ]:
# # # looping through all records for MIT-BIH database
indices_all = list(range(100, 300))
indices_checked = []
indices_paced = [102, 104, 107, 217] # paced beats, need to be removed
indices_AFib = []
indices_AFlu = []
dfs_events = []

for idx in indices_all:
    if idx in indices_paced:
        continue
    try :
        _, df_ann = extract_to_df(idx)
        if df_ann['AuxNote'].str.contains('AFIB').any():
            indices_AFib.append(idx)
        if df_ann['AuxNote'].str.contains('AFL').any():
            indices_AFlu.append(idx)
        indices_checked.append(idx)
        dfs_events.append(df_ann)
    except:
        pass
print(f"Checked {len(indices_checked)} records")
print(f"Found {len(indices_AFib)} with AFIB annotations: ", indices_AFib)
print(f"Found {len(indices_AFlu)} with AFL annotations: ", indices_AFlu)
df_events = pd.concat(dfs_events, ignore_index=True)
print(df_events['AuxNote'].value_counts())


In [ ]:
idx = 10
record = extract_record(idx)
record.to_dataframe()
# df_wave, df_ann = extract_to_df(idx)
# df_wave, df_ann

from extract import extract_record, extract_annotation
df_ann = extract_annotation(idx)
df_ann

In [ ]:
df_ann['AuxNote'].unique()

In [ ]:
# # # looping through all records for Long-Term-AF database
indices_all = list(range(0, 210))
indices_checked = []
wave_columns = []
annotations = set()
for idx in indices_all:
    try :
        record = extract_record(idx)
        df_wave = record.to_dataframe()
        wave_columns.append(df_wave.columns)
        df_ann = extract_annotation(idx)
        annotations.update(df_ann['AuxNote'].unique())
    except Exception as e:
        print(f"Error with record {idx} due to error: {e}")
        pass

In [ ]:
annotations

### Study How to Cut

In [ ]:
from extract import extract_to_df, extract_record
from extract import extract_annotation
from load import extract_event, AF_MIN_SIZE
idx = 222
_, df_ann = extract_to_df(idx)
events = extract_event(df_ann)

### Load the dataset

In [ ]:
from load import SAMPLE_SIZE
data = load_dataset(indices = indices_checked, stride = SAMPLE_SIZE)
#ata = load_dataset(range(100, 300), stride = SAMPLE_SIZE // 3) # stride = 1/3 of sample size = 1/3 of 15s = 5s

### Preporcess

In [ ]:
from preprocess import bandpass_filtering, resample_data
from analysis import plot_wave

idx=1

plot_wave(data, idx, title="Before Preprocessing")

data = resample_data(data)
plot_wave(data, idx, title="After Resampling to 200hz")

data = bandpass_filtering(data)
plot_wave(data, idx, title="After Bandpass filtering")

In [ ]:
from preprocess import process_data
X, y = process_data(data)

# # check the distribution of label y
print("Distribution of labels:\n", pd.Series(y).value_counts())
# print(y.value_counts(normalize=True))

In [ ]:
# split the X,y data into train and test sets using stratified sampling on y
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print("Train label counts:\n", pd.Series(y_train).value_counts())
print("Test label counts:\n", pd.Series(y_test).value_counts())

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
class_weights = dict(enumerate(compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)))  
#{0: 0.3712482536011948, 1: 3.4853007688828583, 2: 51.373333333333335}
#class_weights =  {0: 0.3712482536011948, 1: 3.4853007688828583 * 2, 2: 51.373333333333335 * 4}
print("Class Weights:", class_weights)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Convolution1D, MaxPool1D, Flatten, BatchNormalization
from tensorflow.keras.optimizers import Adam

def network(X_train, y_train, X_test, y_test, filter_sizes={1:4, 2:4, 3:4}, kernel_sizes={1:6, 2:3, 3:3}, learning_rate=0.001, epochs=40, modelCheckPoint=False, class_weights=None):    
    im_shape=(X_train.shape[1], 1)
    inputs_cnn=Input(shape=(im_shape), name='inputs_cnn')
    conv1_1=Convolution1D(filters=filter_sizes[1], kernel_size=(kernel_sizes[1]), activation='relu')(inputs_cnn)
    conv1_1=BatchNormalization()(conv1_1)
    pool1=MaxPool1D(pool_size=(3), strides=(2), padding="same")(conv1_1)
    conv2_1=Convolution1D(filters=filter_sizes[2], kernel_size=(kernel_sizes[2]), activation='relu')(pool1)
    conv2_1=BatchNormalization()(conv2_1)
    pool2=MaxPool1D(pool_size=(2), strides=(2), padding="same")(conv2_1)
    conv3_1=Convolution1D(filters=filter_sizes[3], kernel_size=(kernel_sizes[3]), activation='relu')(pool2)
    conv3_1=BatchNormalization()(conv3_1)
    pool3=MaxPool1D(pool_size=(2), strides=(2), padding="same")(conv3_1)
    flatten=Flatten()(pool3)
    dense_end1 = Dense(units=filter_sizes[3], activation='relu')(flatten)
    dense_end2 = Dense(units=filter_sizes[3], activation='relu')(dense_end1)
    main_output = Dense(units=3, activation='softmax', name='main_output')(dense_end2)

    optimizer = Adam(learning_rate=learning_rate)
    model = Model(inputs=inputs_cnn, outputs=main_output)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy']) # for integer encoding

    # define callbacks for early stopping and saving the best model with lowest validation loss
    callbacks = [EarlyStopping(monitor='val_loss', patience=8)]
    if modelCheckPoint:
        callbacks.append(ModelCheckpoint(filepath='best_model.h5', monitor='val_loss', save_best_only=True))
    history=model.fit(X_train, y_train, epochs=epochs, 
                      callbacks=callbacks, batch_size=32, validation_data=(X_test,y_test),
                      class_weight=class_weights) # assign class weights to the model 
    if modelCheckPoint:
        model.load_weights('best_model.h5')
    return (model, history)

In [ ]:
model,history=network(X_train, y_train, X_test, y_test, 
                      filter_sizes={1:16, 2:32, 3:64}, kernel_sizes={1:2, 2:2, 3:2}, 
                      learning_rate=0.0001, epochs=40, 
                      class_weights=class_weights)

In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv1D, GlobalAveragePooling1D, Dropout, BatchNormalization, ReLU
def network2(X_train, y_train, X_test, y_test, 
             num_classes=3, learning_rate=0.001, epochs=40, 
             modelCheckPoint=False, class_weights=None):    
    inputs_cnn = Input(shape=(X_train.shape[1], 1))

    # Block 1
    x = Conv1D(16, kernel_size=7, padding='same')(inputs_cnn)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    # Block 2
    x = Conv1D(32, kernel_size=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(0.2)(x)
    x = Conv1D(32, kernel_size=3, padding='same')(x)

    # Block 3
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv1D(64, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Dropout(0.2)(x)
    x = Conv1D(64, kernel_size=3, padding='same')(x)

    # Final block
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = GlobalAveragePooling1D()(x)

    # Output layer
    outputs_cnn = Dense(num_classes, activation='softmax')(x)

    optimizer = Adam(learning_rate=learning_rate)
    model = Model(inputs=inputs_cnn, outputs=outputs_cnn)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy']) # for integer encoding

    # define callbacks for early stopping and saving the best model with lowest validation loss
    callbacks = [EarlyStopping(monitor='val_loss', patience=8)]
    if modelCheckPoint:
        callbacks.append(ModelCheckpoint(filepath='best_model.h5', monitor='val_loss', save_best_only=True))
    history=model.fit(X_train, y_train, epochs=epochs, 
                      callbacks=callbacks, batch_size=32, validation_data=(X_test, y_test),
                      class_weight=class_weights) # assign class weights to the model 
    if modelCheckPoint:
        model.load_weights('best_model.h5')
    return (model, history)

In [ ]:
model2,history2=network2(X_train, y_train, X_test, y_test,
                      num_classes=3, learning_rate=0.0001, epochs=150, 
                      class_weights=class_weights)

### Error Analysis

In [ ]:
from analysis import plot_accuracy_and_loss, plot_confusion_matrix
from sklearn.metrics import confusion_matrix

In [ ]:
plot_accuracy_and_loss(history2, X_test, y_test, model2)

In [ ]:
y_pred=model2.predict(X_test)
cnf_matrix = confusion_matrix(y_test, y_pred.argmax(axis=1))
plot_confusion_matrix(cnf_matrix, classes=['N', 'AFib', 'AFlu'], normalize=True)